Описание: Краткое введение и назначение ноутбука.
# Bounce-only: подбор множителей и методов уровней

Этот ноутбук содержит только логику и подбор для стратегии отскока (Buy/Sell Limit).
Запускайте ячейки сверху вниз.

In [ ]:
# Описание: импорт библиотек и подавление частых предупреждений
import os
import warnings
from itertools import product
import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy
from tqdm.auto import tqdm

warnings.filterwarnings(
    "ignore",
    message=".*contingent SL/TP order would execute in the same bar.*",
)
warnings.filterwarnings(
    "ignore",
    message=".*Broker canceled the order due to insufficient margin.*",
)


In [ ]:
# Описание: конфигурация эксперимента и сетки параметров для перебора
CONFIG = {
    'data_file': 'EURUSD_H1_2020-01-01_2025-12-31.csv',
    'atr_period': 14,
    'split_date': '2025-01-01',
    'cash': 100_000,
    'leverage': 100,
    'commission': 0.00002,
    'min_trades': 30,
    'top_n': 10,
}
BUFFER_GRID = [0.1, 0.15, 0.2, 0.25]
SL_GRID = [0.5, 1.0, 1.5, 2.0]
TP_GRID = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0]
LEVEL_METHOD_NAMES = {0: 'Camarilla', 1: 'Pivot (классика)', 2: 'DeMark'}
SUP_COLS = {0: ['cam_s1','cam_s2','cam_s3','cam_s4'], 1: ['piv_s1','piv_s2','piv_s3'], 2: ['dem_s1']}
RES_COLS = {0: ['cam_r1','cam_r2','cam_r3','cam_r4'], 1: ['piv_r1','piv_r2','piv_r3'], 2: ['dem_r1']}

In [ ]:
# Описание: функции вычисления дневных уровней (Camarilla, Pivot, DeMark)
def camarilla_levels(d1):
    h,l,c = d1['High'], d1['Low'], d1['Close']
    hl = h-l
    out = pd.DataFrame(index=d1.index)
    for i,div in enumerate([12,6,4,2], start=1):
        out[f'cam_r{i}'] = c + hl * 1.1 / div
        out[f'cam_s{i}'] = c - hl * 1.1 / div
    return out
def pivot_levels(d1):
    h,l,c = d1['High'], d1['Low'], d1['Close']
    p = (h+l+c)/3
    out = pd.DataFrame(index=d1.index)
    out['piv_r1'] = 2*p - l
    out['piv_s1'] = 2*p - h
    out['piv_r2'] = p + (h-l)
    out['piv_s2'] = p - (h-l)
    out['piv_r3'] = h + 2*(p-l)
    out['piv_s3'] = l - 2*(h-p)
    return out
def demark_levels(d1):
    h,l,c,o = d1['High'], d1['Low'], d1['Close'], d1['Open']
    x = pd.Series(np.where(c<o, h+2*l+c, np.where(c>o, 2*h + l + c, h + l + 2*c)), index=d1.index)
    out = pd.DataFrame(index=d1.index)
    out['dem_r1'] = x/2 - l
    out['dem_s1'] = x/2 - h
    return out

In [ ]:
# Описание: загрузка CSV, вычисление дневных уровней и ATR
def load_and_prepare(csv_path, cfg):
    df = pd.read_csv(csv_path)
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time').sort_index()
    df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','tick_volume':'Volume'})[["Open","High","Low","Close","Volume"]]
    off = pd.Timedelta(hours=0)
    day = (df.index + off).normalize()
    d1 = df.groupby(day).agg(Open=('Open','first'), High=('High','max'), Low=('Low','min'), Close=('Close','last'))
    lev = pd.concat([camarilla_levels(d1), pivot_levels(d1), demark_levels(d1)], axis=1).shift(1)
    lev.index.name = '_day'
    df = df.assign(_day=day).join(lev, on='_day').drop(columns='_day')
    prev = df['Close'].shift(1)
    tr = pd.concat([df['High']-df['Low'], (df['High']-prev).abs(), (df['Low']-prev).abs()], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(cfg['atr_period']).mean()
    return df

In [ ]:
# Описание: класс стратегии отскока — логика открытия лимитных ордеров
class BounceStrategy(Strategy):
    level_method = 0
    buffer_atr_s = 0.5
    buffer_atr_r = 0.5
    sl_atr_s = 2.0
    sl_atr_r = 2.0
    tp_atr_s = 2.0
    tp_atr_r = 2.0
    long_only = False
    short_only = False
    risk_pct = 0.02            # доля equity на сделку
    margin_use = 0.5           # макс. доля доступной маржи на один ордер
    max_open_positions = 1     # макс. одновременных позиций + ожидающих ордеров
    max_lot_size = 0.1         # жёсткий потолок: 0.1 лота
    lot_units = 100_000
    max_position_size = max_lot_size * lot_units
    def init(self):
        df = self.data.df
        self._i = 0
        self._close = df['Close'].to_numpy()
        self._atr = df['ATR'].to_numpy()
        self._sup = {c: df[c].to_numpy() for c in SUP_COLS[self.level_method]}
        self._res = {c: df[c].to_numpy() for c in RES_COLS[self.level_method]}
        self._open_positions = 0
    def _cap_size(self, size):
        return min(max(1, int(round(size))), int(self.max_position_size))
    def _position_size(self, entry, risk):
        available = getattr(self, 'equity', CONFIG['cash'])
        size = max(1, int(round((available * self.risk_pct) / risk)))
        size = self._cap_size(size)
        # ограничение по доступной марже (margin_available — у брокера):
        # size * entry <= margin_use * margin_available * leverage
        broker = getattr(self, '_broker', None)
        margin_available = getattr(broker, 'margin_available', None)
        if margin_available is not None and entry > 0:
            max_size = int(margin_available * CONFIG['leverage'] * self.margin_use / entry)
            if max_size < 1:
                return 0  # маржи не хватает даже на 1 единицу — ордер не ставим
            size = min(size, max_size)
        return max(1, size)
    def _active_count(self):
        # сколько одновременно занято: открытая позиция + ожидающие ордера
        pos = getattr(self, 'position', None)
        n_pos = 1 if (pos is not None and pos.size != 0) else 0
        n_orders = len(getattr(self, 'orders', ()))
        return n_pos + n_orders
    def next(self):
        i = self._i; self._i += 1
        atr = self._atr[i]
        if not np.isfinite(atr) or atr<=0: return
        close = self._close[i]
        if self._active_count() >= self.max_open_positions: return
        if not self.short_only:
            sup = None
            for arr in self._sup.values():
                v = arr[i];
                if v <= close and (sup is None or v>sup): sup = v
            if sup is not None:
                entry = sup + self.buffer_atr_s * atr
                risk = self.sl_atr_s * atr
                size = self._position_size(entry, risk)
                if size > 0 and close>entry: self.buy(size=size, limit=entry, sl=entry-self.sl_atr_s*atr, tp=entry+self.tp_atr_s*atr)
        if not self.long_only:
            res = None
            for arr in self._res.values():
                v = arr[i];
                if v >= close and (res is None or v<res): res = v
            if res is not None:
                entry = res - self.buffer_atr_r * atr
                risk = self.sl_atr_r * atr
                size = self._position_size(entry, risk)
                if size > 0 and close<entry: self.sell(size=size, limit=entry, sl=entry+self.sl_atr_r*atr, tp=entry-self.tp_atr_r*atr)


In [ ]:
# Описание: функции для статистической строки, перебора сеток и форматирования топ-таблицы


# Все доступные метрики backtesting.py (кроме служебных _strategy/_equity_curve/_trades)
METRICS = [
    'Start', 'End', 'Duration',
    'Exposure Time [%]',
    'Equity Final [$]',
    'Equity Peak [$]',
    'Return [%]',
    'Buy & Hold Return [%]',
    'Return (Ann.) [%]',
    'Volatility (Ann.) [%]',
    'CAGR [%]',
    'Sharpe Ratio',
    'Sortino Ratio',
    'Calmar Ratio',
    'Alpha [%]',
    'Beta',
    'Max. Drawdown [%]',
    'Avg. Drawdown [%]',
    'Max. Drawdown Duration',
    'Avg. Drawdown Duration',
    'Win Rate [%]',
    'Best Trade [%]',
    'Worst Trade [%]',
    'Avg. Trade [%]',
    'Max. Trade Duration',
    'Avg. Trade Duration',
    'Profit Factor',
    'Expectancy [%]',
    'SQN',
    'Kelly Criterion',
]
TRADES_KEY = 'Trades'  # короткое имя для '# Trades'

def stats_row(s, params):
    row = dict(params)
    for m in METRICS:
        row[m] = s.get(m, np.nan)
    row[TRADES_KEY] = s.get('# Trades', len(s.get('_trades', [])))
    return row

def grid_search(bt_obj, grids, fixed, cfg):
    keys = list(grids)
    combos = list(product(*grids.values()))
    rows = []
    for combo in tqdm(combos, desc='Перебор', unit='комб'):
        params = dict(zip(keys, combo))
        params.update({k: v[0] for k,v in fixed.items()})
        s = bt_obj.run(**params)
        rows.append(stats_row(s, params))
    out = pd.DataFrame(rows)
    out = out[out[TRADES_KEY] >= cfg['min_trades']]
    if out.empty: out = pd.DataFrame(rows)
    return out.sort_values('Sharpe Ratio', ascending=False).reset_index(drop=True)

def top_table(rows, cfg):
    d = rows.head(cfg['top_n']).copy()
    d.insert(1, 'Метод', d['level_method'].map(LEVEL_METHOD_NAMES))
    cols = ['Метод'] + METRICS + [TRADES_KEY]
    return d[cols]


In [ ]:
# Описание: загрузить файл, отфильтровать по датам и подготовить Backtest объекты
if 'CONFIG' not in globals():
    raise NameError("CONFIG не определён. Запустите ячейку с конфигурацией (CONFIG) выше.")

DATA_PATH = next((p for p in ['content/' + CONFIG['data_file'], CONFIG['data_file']] if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Data file not found: {CONFIG['data_file']}")

df = load_and_prepare(DATA_PATH, CONFIG)
df = df.loc['2024':'2025']
split = pd.Timestamp(CONFIG['split_date'])
df_train = df[df.index < split]
df_test = df[df.index >= split]
bt_train = Backtest(df_train, BounceStrategy, cash=CONFIG['cash'], commission=CONFIG['commission'], margin=1/CONFIG['leverage'], finalize_trades=True)
bt_test = Backtest(df_test, BounceStrategy, cash=CONFIG['cash'], commission=CONFIG['commission'], margin=1/CONFIG['leverage'], finalize_trades=True)


In [ ]:
# Проход: ОТСКОК ОТ ПОДДЕРЖКИ — подбор S (Buy Limit)
r1 = grid_search(bt_train, grids={'level_method': range(3), 'buffer_atr_s': BUFFER_GRID, 'sl_atr_s': SL_GRID, 'tp_atr_s': TP_GRID}, fixed={'buffer_atr_r':[0.5],'sl_atr_r':[1.0],'tp_atr_r':[2.0],'long_only':[True],'short_only':[False]}, cfg=CONFIG)
best1 = r1.iloc[0]
display(top_table(r1, CONFIG))

Перебор: 100%|██████████| 288/288 [00:16<00:00, 17.67комб/s]


,Метод,Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],...,Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,Trades
0,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,4.384838,100366.902692,100366.902692,0.366903,-6.289445,0.387805,...,0.822853,-0.208551,0.092505,2 days 06:00:00,0 days 09:00:00,2.854336,0.092795,2.316218,0.263204,37
1,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.967234,100306.335287,100306.335287,0.306335,-6.289445,0.329088,...,0.685044,-0.208551,0.073368,2 days 06:00:00,0 days 08:00:00,2.499656,0.073586,2.175001,0.245868,39
2,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.919049,100266.141786,100266.141786,0.266142,-6.289445,0.290122,...,0.822967,-0.202968,0.075408,2 days 04:00:00,0 days 08:00:00,2.395549,0.075700,1.775855,0.211405,33
3,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.003534,100238.863780,100249.983613,0.238864,-6.289445,0.263677,...,0.547235,-0.208551,0.060181,2 days 07:00:00,0 days 07:00:00,2.223211,0.060351,1.969696,0.237859,37
4,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.437199,100229.346024,100229.346024,0.229346,-6.289445,0.254450,...,0.823081,-0.197385,0.071561,2 days 06:00:00,0 days 08:00:00,2.358595,0.071851,1.608266,0.210613,30
5,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,2.168326,100146.386910,100164.010731,0.146387,-6.289445,0.174022,...,0.409540,-0.197385,0.041485,2 days 06:00:00,0 days 05:00:00,1.887020,0.041606,1.515985,0.213175,33
6,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.629939,100188.404093,100188.404093,0.188404,-6.289445,0.214758,...,0.685139,-0.202968,0.051844,2 days 04:00:00,0 days 08:00:00,1.958403,0.052058,1.443417,0.172243,34
7,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,2.634115,100176.342244,100185.150075,0.176342,-6.289445,0.203064,...,0.547311,-0.202968,0.048457,2 days 04:00:00,0 days 06:00:00,1.942287,0.048625,1.523138,0.185163,34
8,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,2.666238,100149.916993,100171.175884,0.149917,-6.289445,0.177445,...,0.409426,-0.208551,0.038829,2 days 07:00:00,0 days 07:00:00,1.791381,0.038951,1.484036,0.196202,36
9,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,4.047543,100223.326463,100258.173196,0.223326,-6.289445,0.267813,...,0.678573,-0.334687,0.062541,3 days 12:00:00,0 days 13:00:00,1.744497,0.062899,1.336135,0.168695,33


In [ ]:
# Проход: ОТСКОК ОТ СОПРОТИВЛЕНИЯ — подбор R (Sell Limit)
r2 = grid_search(bt_train, grids={'level_method': range(3), 'buffer_atr_r': BUFFER_GRID, 'sl_atr_r': SL_GRID, 'tp_atr_r': TP_GRID}, fixed={'buffer_atr_s':[0.5],'sl_atr_s':[1.0],'tp_atr_s':[2.0],'long_only':[False],'short_only':[True]}, cfg=CONFIG)
best2 = r2.iloc[0]
display(top_table(r2, CONFIG))

Перебор: 100%|██████████| 288/288 [00:14<00:00, 19.34комб/s]


,Метод,Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],...,Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,Trades
0,Camarilla,2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.533569,100262.025151,100262.025151,0.262025,-6.289445,0.186741,...,0.627074,-0.188720,0.077862,2 days 15:00:00,0 days 08:00:00,2.804395,0.078142,1.779465,0.214735,30
1,Camarilla,2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.565692,100240.922122,100240.922122,0.240922,-6.289445,0.165545,...,0.634149,-0.192383,0.071558,2 days 14:00:00,0 days 08:00:00,2.668918,0.071840,1.631205,0.229611,30
2,Camarilla,2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.581754,100222.460228,100222.460228,0.222460,-6.289445,0.154464,...,0.589290,-0.196047,0.063891,2 days 14:00:00,0 days 08:00:00,2.402309,0.064159,1.516876,0.207513,31
3,Camarilla,2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,2.987472,100091.912717,100092.856502,0.091913,-6.289445,0.043383,...,0.440968,-0.196047,0.027190,2 days 05:00:00,0 days 06:00:00,1.600116,0.027337,0.861916,0.112974,30


In [ ]:
# Проход 3: выбор метода для ЛОНГ и ШОРТ при найденных множителях
r3_long = grid_search(bt_train, grids={'level_method': range(3)}, fixed={'buffer_atr_s':[best1['buffer_atr_s']],'sl_atr_s':[best1['sl_atr_s']],'tp_atr_s':[best1.get('tp_atr_s',2.0)],'buffer_atr_r':[0.5],'sl_atr_r':[1.0],'tp_atr_r':[2.0],'long_only':[True],'short_only':[False]}, cfg=CONFIG)
best_long = r3_long.iloc[0]
display(top_table(r3_long, CONFIG))
r3_short = grid_search(bt_train, grids={'level_method': range(3)}, fixed={'buffer_atr_s':[0.5],'sl_atr_s':[1.0],'tp_atr_s':[2.0],'buffer_atr_r':[best2['buffer_atr_r']],'sl_atr_r':[best2['sl_atr_r']],'tp_atr_r':[best2.get('tp_atr_r',2.0)],'long_only':[False],'short_only':[True]}, cfg=CONFIG)
best_short = r3_short.iloc[0]
display(top_table(r3_short, CONFIG))
# Итоговый вывод
print('--- Лучшие методы (Bounce) ---')
print('ОТСКОК ОТ ПОДДЕРЖКИ :', LEVEL_METHOD_NAMES[int(best_long['level_method'])], 'params S:', {'buffer_atr_s':float(best_long['buffer_atr_s']),'sl_atr_s':float(best_long['sl_atr_s']),'tp_atr_s':float(best_long.get('tp_atr_s',2.0))})
print('ОТСКОК ОТ СОПРОТИВЛЕНИЯ :', LEVEL_METHOD_NAMES[int(best_short['level_method'])], 'params R:', {'buffer_atr_r':float(best_short['buffer_atr_r']),'sl_atr_r':float(best_short['sl_atr_r']),'tp_atr_r':float(best_short.get('tp_atr_r',2.0))})

Перебор: 100%|██████████| 3/3 [00:00<00:00, 16.30комб/s]


,Метод,Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],...,Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,Trades
0,Pivot (классика),2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,4.384838,100366.902692,100366.902692,0.366903,-6.289445,0.387805,...,0.822853,-0.208551,0.092505,2 days 06:00:00,0 days 09:00:00,2.854336,0.092795,2.316218,0.263204,37
1,Camarilla,2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,5.573402,100078.879501,100204.452285,0.078880,-6.289445,0.130699,...,0.571451,-0.407090,0.009717,2 days 20:00:00,0 days 05:00:00,1.151195,0.009912,0.429084,0.036557,75


Перебор: 100%|██████████| 3/3 [00:00<00:00, 17.69комб/s]


,Метод,Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],...,Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,Trades
0,Camarilla,2024-01-02,2024-12-31 23:00:00,364 days 23:00:00,3.533569,100262.025151,100262.025151,0.262025,-6.289445,0.186741,...,0.627074,-0.18872,0.077862,2 days 15:00:00,0 days 08:00:00,2.804395,0.078142,1.779465,0.214735,30


--- Лучшие методы (Bounce) ---
ОТСКОК ОТ ПОДДЕРЖКИ : Pivot (классика) params S: {'buffer_atr_s': 0.25, 'sl_atr_s': 0.5, 'tp_atr_s': 3.0}
ОТСКОК ОТ СОПРОТИВЛЕНИЯ : Camarilla params R: {'buffer_atr_r': 0.15, 'sl_atr_r': 0.5, 'tp_atr_r': 4.0}
